# Pandas — Merging and Joining

> **Repo:** Python_Libraries | **Library:** Pandas  
> **Notebook:** 07_Merging_and_Joining  

---

# 07 -- Merging and Joining In Pandas

Combining data from multiple sources is one of the most common tasks in data analysis.

Pandas provides three main tools for this:

| Function | Best For |
|---|---|
| `pd.merge()` | SQL-style joins on columns or index |
| `pd.concat()` | Stacking DataFrames row-wise or column-wise |
| `df.join()` | Quick index-based joins |

**When would you use this?**
- Combining a sales table with a customer table
- Appending new monthly data to existing records
- Enriching a dataset with lookup/reference data

**Dataset used:** A simulated e-commerce system with 4 related tables —
`customers`, `orders`, `order_items`, and `products`.

## Section 01: Importing Libraries and Creating Dataset

In [4]:
import pandas as pd
import numpy as np

# ── Table 1: Customers ──────────────────────────────
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name':        ['Amit', 'Priya', 'Ravi', 'Sneha', 'Karan',
                    'Divya', 'Rohit', 'Meena', 'Arjun', 'Pooja'],
    'city':        ['Delhi', 'Mumbai', 'Bangalore', 'Pune', 'Hyderabad',
                    'Chennai', 'Kolkata', 'Jaipur', 'Ahmedabad', 'Surat'],
    'segment':     ['Retail', 'Wholesale', 'Retail', 'Retail', 'Wholesale',
                    'Retail', 'Wholesale', 'Retail', 'Retail', 'Wholesale']
})

# ── Table 2: Orders ─────────────────────────────────
orders = pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105, 106, 107, 108,
                    109, 110, 111, 112, 113, 114, 115],
    'customer_id': [1, 2, 1, 3, 4, 5, 6, 7,
                    8, 2, 3, 5, 1, 7, 99], 
    'order_date':  pd.to_datetime([
                    '2024-01-05', '2024-01-08', '2024-01-15', '2024-02-01',
                    '2024-02-14', '2024-02-20', '2024-03-01', '2024-03-10',
                    '2024-03-18', '2024-04-02', '2024-04-11', '2024-04-25',
                    '2024-05-03', '2024-05-17', '2024-06-01']),
    'status':      ['Delivered', 'Delivered', 'Shipped', 'Delivered', 'Cancelled',
                    'Delivered', 'Shipped', 'Delivered', 'Delivered', 'Cancelled',
                    'Delivered', 'Shipped', 'Delivered', 'Delivered', 'Pending']
})

# ── Table 3: Order Items (20 rows) ───────────────────────────
order_items = pd.DataFrame({
    'item_id':    list(range(1, 21)),
    'order_id':   [101, 101, 102, 103, 104, 104, 105, 106,
                   107, 107, 108, 109, 110, 110, 111, 112,
                   113, 113, 114, 115],
    'product_id': [1, 3, 2, 5, 1, 4, 6, 2,
                   3, 7, 8, 5, 1, 6, 4, 3,
                   2, 7, 8, 1],
    'quantity':   [1, 2, 1, 3, 1, 1, 2, 4,
                   1, 1, 2, 1, 3, 1, 2, 1,
                   1, 2, 1, 2],
    'unit_price': [75000, 1500, 500, 3000, 75000, 18000, 2500, 500,
                   1500, 8000, 4500, 3000, 75000, 2500, 18000, 1500,
                   500, 8000, 4500, 75000]
})

# ── Table 4: Products (8 rows) ────────────────────────────────
products = pd.DataFrame({
    'product_id':   [1, 2, 3, 4, 5, 6, 7, 8],
    'product_name': ['Laptop', 'Mouse', 'Keyboard', 'Monitor',
                     'Webcam', 'USB Hub', 'SSD 1TB', 'Headphones'],
    'category':     ['Computing', 'Accessories', 'Accessories', 'Display',
                     'Accessories', 'Accessories', 'Storage', 'Audio'],
    'brand':        ['Dell', 'Logitech', 'HP', 'Samsung',
                     'Zebronics', 'AmazonBasics', 'WD', 'Sony']
})

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)

customers: (10, 4)
orders: (15, 4)
order_items: (20, 5)
products: (8, 4)


## Section 02: `pd.merge()` - The Core Function

`pd.merge()` works like SQL JOINs. It combines two DataFrames based on a common key (column).

#### Syntax:
```python
pd.merge(left, right, how='inner', on='key_column')
```

| `how=` | Keeps |
|---|---|
| `'inner'` | Only matching rows from BOTH tables |
| `'left'` | All rows from left + matching from right |
| `'right'` | All rows from right + matching from left |
| `'outer'` | All rows from BOTH tables |

### 2.1 Inner Join — orders ⟕ customers

**Business Question:** 
> *Which orders were placed by known customers?*

In [20]:
# INNER JOIN — orders with customers
# Only rows where customer_id exists in BOTH tables

inner_join = pd.merge(orders, customers, on='customer_id', how='inner')

print("Orders total:", len(orders))
print("After INNER JOIN with customers:", len(inner_join))

inner_join

Orders total: 15
After INNER JOIN with customers: 14


,order_id,customer_id,order_date,status,name,city,segment
0,101,1,2024-01-05,Delivered,Amit,Delhi,Retail
1,102,2,2024-01-08,Delivered,Priya,Mumbai,Wholesale
2,103,1,2024-01-15,Shipped,Amit,Delhi,Retail
3,104,3,2024-02-01,Delivered,Ravi,Bangalore,Retail
4,105,4,2024-02-14,Cancelled,Sneha,Pune,Retail
5,106,5,2024-02-20,Delivered,Karan,Hyderabad,Wholesale
6,107,6,2024-03-01,Shipped,Divya,Chennai,Retail
7,108,7,2024-03-10,Delivered,Rohit,Kolkata,Wholesale
8,109,8,2024-03-18,Delivered,Meena,Jaipur,Retail
9,110,2,2024-04-02,Cancelled,Priya,Mumbai,Wholesale


#### Interpretation:

All the orders from `orders` table were included except the order **115** which was ordered by a ghost customer **99** which doesn't exist in `customers` table.

### 2.2 Left Join — Keep ALL From Left Table, and matching Rows from Right Table

**Business Question:**

> *Show all orders — even those with unrecognized customer IDs.*

In [34]:
# LEFT JOIN — all orders retained, NaN where no customer match
left_join = pd.merge(orders, customers, on='customer_id', how='left')

print("Orders total:", len(orders))
print("After LEFT JOIN:", len(left_join), "← all orders kept\n")

# Show the unmatched row clearly
print("Unmatched orders (NaN in customer info):")
left_join[left_join['name'].isna()]

Orders total: 15
After LEFT JOIN: 15 ← all orders kept

Unmatched orders (NaN in customer info):


,order_id,customer_id,order_date,status,name,city,segment
14,115,99,2024-06-01,Pending,NaN,NaN,NaN


#### Interpretation:
- Rows with no match in `customers` will show `NaN` for name, city, segment.
- order_id 115 (customer_id=99) appeared with NaN values.

### 2.3 Right Join -- Keep ALL from Right Table, MATCH from the LEFT Table

**Business Question:**

> *Show all customers — even those who never placed an order.*

In [74]:
# RIGHT JOIN — all customers retained
right_join = pd.merge(orders, customers, on='customer_id', how='right')

print("Customers total:", len(customers))
print("After RIGHT JOIN:", len(right_join))

# Show customers with no orders
print("\nCustomers with NO orders (NaN in order columns):")
print(right_join[right_join['order_id'].isna()])

#Show customers who placed more than one order
print("\nCustomers with more than one order: ")
premium_customers =  right_join.groupby('name').filter(lambda x: x['name'].count() > 1)
premium_customers['name'].unique()

Customers total: 10
After RIGHT JOIN: 16

Customers with NO orders (NaN in order columns):
    order_id  customer_id order_date status   name       city    segment
14       NaN            9        NaT    NaN  Arjun  Ahmedabad     Retail
15       NaN           10        NaT    NaN  Pooja      Surat  Wholesale

Customers with more than one order: 


array(['Amit', 'Priya', 'Ravi', 'Karan', 'Rohit'], dtype=object)

### 2.4 Outer Join — Keep EVERYTHING from both tables

**Business Question:** 

> * Show all orders AND all customers, matched where possible.*

In [82]:
# OUTER JOIN — union of both tables
outer_join = pd.merge(orders, customers, on='customer_id', how='outer')

print("Outer join shape:", outer_join.shape)

# All unmatched rows from either side
print("\nAll unmatched rows (NaN anywhere):")
outer_join[outer_join.isna().any(axis=1)]

Outer join shape: (17, 7)

All unmatched rows (NaN anywhere):


,order_id,customer_id,order_date,status,name,city,segment
14,NaN,9,NaT,NaN,Arjun,Ahmedabad,Retail
15,NaN,10,NaT,NaN,Pooja,Surat,Wholesale
16,115.0,99,2024-06-01,Pending,NaN,NaN,NaN


## Section 3: Merging on Multiple Keys & `left_on` / `right_on`

### 3.1 Merging on Multiple Keys

Sometimes a single column isn't enough to uniquely identify a match.

We pass a **list** to `on=` to merge on multiple columns simultaneously.

**Business Question:**

> *Find all delivered orders along with their customer details.*

(Match on both `customer_id` AND `status` to filter at merge time)

**Note:**

- This is different from filtering after merge -
- The join itself only connects rows where BOTH keys match simultaneously.

In [7]:
# Create a reference table of "approved" customer-status combinations
# Simulates a business rule: only these customer+status combos are valid
approved = pd.DataFrame({
    'customer_id': [1, 2, 3, 5, 7],
    'status':      ['Delivered', 'Delivered', 'Delivered', 'Delivered', 'Delivered']
})

# Merge on TWO keys — both must match
multi_key_merge = pd.merge(orders, approved, on=['customer_id', 'status'], how='inner')

print("Orders matching approved customer+status combos:")
print(multi_key_merge[['order_id', 'customer_id', 'status', 'order_date']])
print("\nTotal matched:", len(multi_key_merge))

Orders matching approved customer+status combos:
   order_id  customer_id     status order_date
0       101            1  Delivered 2024-01-05
1       102            2  Delivered 2024-01-08
2       104            3  Delivered 2024-02-01
3       106            5  Delivered 2024-02-20
4       108            7  Delivered 2024-03-10
5       111            3  Delivered 2024-04-11
6       113            1  Delivered 2024-05-03
7       114            7  Delivered 2024-05-17

Total matched: 8


### 3.2 `left_on` / `right_on`: Joining on differently Named Keys

In real-world datasets, the same logical key often has **different column names** in different tables.

Therefore, we use `left_on` and `right_on` instead of `on=`.

**Business Question:**

> The order_items table uses `order_id` but imagine a shipments table uses `ref_order_id` — we still need to join them.

In [12]:
# Simulating a shipments table where the key has a different name
shipments = pd.DataFrame({
    'shipment_id':  [201, 202, 203, 204, 205],
    'ref_order_id': [101, 104, 107, 111, 113],   # ← different name!
    'carrier':      ['FedEx', 'BlueDart', 'Delhivery', 'DTDC', 'FedEx'],
    'shipped_on':   pd.to_datetime(['2024-01-06', '2024-02-02',
                                    '2024-03-02', '2024-04-12', '2024-05-04'])
})

# Join orders → shipments using left_on / right_on
orders_shipped = pd.merge(
    orders, shipments,
    left_on='order_id',      # key name in LEFT table
    right_on='ref_order_id', # key name in RIGHT table
    how='left'
)

print("Orders with shipment info (left join):\n")
print(orders_shipped[['order_id', 'status', 'shipment_id', 
                       'carrier', 'shipped_on']].to_string(index=False))
print(f"\nOrders with no shipment yet: {orders_shipped['shipment_id'].isna().sum()}")

Orders with shipment info (left join):

 order_id    status  shipment_id   carrier shipped_on
      101 Delivered        201.0     FedEx 2024-01-06
      102 Delivered          NaN       NaN        NaT
      103   Shipped          NaN       NaN        NaT
      104 Delivered        202.0  BlueDart 2024-02-02
      105 Cancelled          NaN       NaN        NaT
      106 Delivered          NaN       NaN        NaT
      107   Shipped        203.0 Delhivery 2024-03-02
      108 Delivered          NaN       NaN        NaT
      109 Delivered          NaN       NaN        NaT
      110 Cancelled          NaN       NaN        NaT
      111 Delivered        204.0      DTDC 2024-04-12
      112   Shipped          NaN       NaN        NaT
      113 Delivered        205.0     FedEx 2024-05-04
      114 Delivered          NaN       NaN        NaT
      115   Pending          NaN       NaN        NaT

Orders with no shipment yet: 10


#### Key Takeaways

| Scenario | Use |
|---|---|
| Same key name in both tables | `on='col'` or `on=['col1','col2']` |
| Different key names | `left_on='x', right_on='y'` |
| Multiple matching conditions | `on=['col1', 'col2']` |

> When using `left_on`/`right_on`, both key columns appear in the result.
> 
> Drop the redundant one with `.drop(columns=['ref_order_id'])`, if needed

## Section 4: `suffixes` — Handling Overlapping Column Names

When two tables share a column name that is NOT the join key, Pandas automatically adds `_x` (left) and `_y` (right) as suffixes.

We can customize these with the `suffixes=` parameter for clarity.

**Business Question:**

> Merge `order_items` with `products` — both have a `unit_price` style column.

Let's engineer a price conflict scenario to see suffixes in action.

In [25]:
# Add a 'listed_price' column to products to simulate overlap
products_with_price = products.copy()
products_with_price['unit_price'] = [72000, 450, 1400, 17500, 2400, 480, 7800, 4200]
# ↑ These are catalogue prices — slightly different from actual order prices

# Merge — both tables have 'unit_price' → collision!
# Without custom suffixes:
merged_default = pd.merge(order_items, products_with_price, on='product_id', how='left')
print("Default suffixes (_x, _y):")
print(merged_default[['item_id', 'product_name', 'unit_price_x', 'unit_price_y']].head(6))

print("\n" + "─"*55)

# With custom suffixes — much more readable
merged_custom = pd.merge(
    order_items, products_with_price,
    on='product_id',
    how='left',
    suffixes=('_charged', '_listed')   # ← meaningful names
)
print("\nCustom suffixes (_charged, _listed):")
print(merged_custom[['item_id', 'product_name', 
                      'unit_price_charged', 'unit_price_listed']].head(6))

# Bonus: find items where charged price differs from listed price
merged_custom['price_diff'] = merged_custom['unit_price_charged'] - merged_custom['unit_price_listed']
print("\nItems with price discrepancy:")
merged_custom[merged_custom['price_diff'] != 0][
    ['item_id', 'product_name', 'unit_price_charged', 'unit_price_listed', 'price_diff']
]

Default suffixes (_x, _y):
   item_id product_name  unit_price_x  unit_price_y
0        1       Laptop         75000         72000
1        2     Keyboard          1500          1400
2        3        Mouse           500           450
3        4       Webcam          3000          2400
4        5       Laptop         75000         72000
5        6      Monitor         18000         17500

───────────────────────────────────────────────────────

Custom suffixes (_charged, _listed):
   item_id product_name  unit_price_charged  unit_price_listed
0        1       Laptop               75000              72000
1        2     Keyboard                1500               1400
2        3        Mouse                 500                450
3        4       Webcam                3000               2400
4        5       Laptop               75000              72000
5        6      Monitor               18000              17500

Items with price discrepancy:


,item_id,product_name,unit_price_charged,unit_price_listed,price_diff
0,1,Laptop,75000,72000,3000
1,2,Keyboard,1500,1400,100
2,3,Mouse,500,450,50
3,4,Webcam,3000,2400,600
4,5,Laptop,75000,72000,3000
5,6,Monitor,18000,17500,500
6,7,USB Hub,2500,480,2020
7,8,Mouse,500,450,50
8,9,Keyboard,1500,1400,100
9,10,SSD 1TB,8000,7800,200


#### Key Takeaways:

| Situation | Result |
|---|---|
| No overlap in non-key columns | No suffixes added |
| Overlap exists, no `suffixes=` | Pandas adds `_x`, `_y` automatically |
| Overlap exists, custom `suffixes=` | Your chosen labels — always prefer this |

**Best practice:**

> Always set `suffixes=` when merging tables that might share column names.

> `_x/_y` in production code is a red flag.

## Section 5: `pd.concat()` — Stacking DataFrames

`pd.merge()` joins tables **horizontally** (adding columns).

`pd.concat()` stacks tables **vertically** (adding rows) or horizontally.

**Common Use Cases:**

- Appending monthly sales data into one annual table
- Combining train/test datasets
- Stacking results from multiple API calls

**Syntax:**
```python
pd.concat([df1, df2], axis=0)   # row-wise (default)
pd.concat([df1, df2], axis=1)   # column-wise
```


In [34]:
# Simulate two quarterly order batches
q1_orders = orders[orders['order_date'] < '2024-04-01'].copy()
q2_orders = orders[orders['order_date'] >= '2024-04-01'].copy()

print(f"Q1 orders: {len(q1_orders)} rows")
print(f"Q2 orders: {len(q2_orders)} rows")

Q1 orders: 9 rows
Q2 orders: 6 rows


### 5.1 Basic concat — row-wise

In [47]:
combined = pd.concat([q1_orders, q2_orders], axis=0)

print(f"\nAfter concat: {len(combined)} rows")
print("\nNote the index — it carries over from original:")
print(combined[['order_id', 'order_date', 'status']])


After concat: 15 rows

Note the index — it carries over from original:
    order_id order_date     status
0        101 2024-01-05  Delivered
1        102 2024-01-08  Delivered
2        103 2024-01-15    Shipped
3        104 2024-02-01  Delivered
4        105 2024-02-14  Cancelled
5        106 2024-02-20  Delivered
6        107 2024-03-01    Shipped
7        108 2024-03-10  Delivered
8        109 2024-03-18  Delivered
9        110 2024-04-02  Cancelled
10       111 2024-04-11  Delivered
11       112 2024-04-25    Shipped
12       113 2024-05-03  Delivered
13       114 2024-05-17  Delivered
14       115 2024-06-01    Pending


### 5.2 ignore_index=True — resets index cleanly

In [49]:
combined_reset = pd.concat([q1_orders, q2_orders], axis=0, ignore_index=True)
print("With ignore_index=True — clean 0-based index:")
print(combined_reset[['order_id', 'order_date', 'status']])

With ignore_index=True — clean 0-based index:
    order_id order_date     status
0        101 2024-01-05  Delivered
1        102 2024-01-08  Delivered
2        103 2024-01-15    Shipped
3        104 2024-02-01  Delivered
4        105 2024-02-14  Cancelled
5        106 2024-02-20  Delivered
6        107 2024-03-01    Shipped
7        108 2024-03-10  Delivered
8        109 2024-03-18  Delivered
9        110 2024-04-02  Cancelled
10       111 2024-04-11  Delivered
11       112 2024-04-25    Shipped
12       113 2024-05-03  Delivered
13       114 2024-05-17  Delivered
14       115 2024-06-01    Pending


### 5.3 `keys= parameter`: adds a label to identify source

In [53]:
combined_labeled = pd.concat(
    [q1_orders, q2_orders],
    axis=0,
    keys=['Q1', 'Q2']   # ← creates a MultiIndex
)
print("With keys= — source labeled in MultiIndex:")
print(combined_labeled[['order_id', 'order_date', 'status']])

# Access just Q2 data
print("\nAccessing Q2 data via MultiIndex:")
print(combined_labeled.loc['Q2'][['order_id', 'status']])

With keys= — source labeled in MultiIndex:
       order_id order_date     status
Q1 0        101 2024-01-05  Delivered
   1        102 2024-01-08  Delivered
   2        103 2024-01-15    Shipped
   3        104 2024-02-01  Delivered
   4        105 2024-02-14  Cancelled
   5        106 2024-02-20  Delivered
   6        107 2024-03-01    Shipped
   7        108 2024-03-10  Delivered
   8        109 2024-03-18  Delivered
Q2 9        110 2024-04-02  Cancelled
   10       111 2024-04-11  Delivered
   11       112 2024-04-25    Shipped
   12       113 2024-05-03  Delivered
   13       114 2024-05-17  Delivered
   14       115 2024-06-01    Pending

Accessing Q2 data via MultiIndex:
    order_id     status
9        110  Cancelled
10       111  Delivered
11       112    Shipped
12       113  Delivered
13       114  Delivered
14       115    Pending


## Section 6: `DataFrame.join()` — Index-Based Joining

`df.join()` is a shortcut for merging on the **index** rather than columns.

Under the hood it calls `pd.merge()`, but with cleaner syntax for index joins.

| Feature | `pd.merge()` | `df.join()` |
|---|---|---|
| Join on columns | ✅ | ❌ (needs `on=` workaround) |
| Join on index | ✅ | ✅ (default) |
| Syntax | Verbose | Concise |

**Business Question:**

> Enrich the products table with category-level metadata (discount rate, return rate) stored in a separate lookup table — indexed by category.

In [57]:
# Category metadata — indexed by category name
category_meta = pd.DataFrame({
    'discount_pct': [10, 5, 8, 12, 6],
    'return_rate':  [3.2, 1.5, 2.8, 4.1, 2.0]
}, index=['Computing', 'Accessories', 'Display', 'Audio', 'Storage'])

print("Category metadata (index-based):")
print(category_meta)

# Set products index to 'category' to align with category_meta
products_indexed = products.set_index('category')

print("\nProducts indexed by category:")
print(products_indexed)

Category metadata (index-based):
             discount_pct  return_rate
Computing              10          3.2
Accessories             5          1.5
Display                 8          2.8
Audio                  12          4.1
Storage                 6          2.0

Products indexed by category:
             product_id product_name         brand
category                                          
Computing             1       Laptop          Dell
Accessories           2        Mouse      Logitech
Accessories           3     Keyboard            HP
Display               4      Monitor       Samsung
Accessories           5       Webcam     Zebronics
Accessories           6      USB Hub  AmazonBasics
Storage               7      SSD 1TB            WD
Audio                 8   Headphones          Sony


In [61]:
# df.join() — joins on index automatically
products_enriched = products_indexed.join(category_meta, how='left')

print("Products enriched with category metadata:\n")
print(products_enriched)

Products enriched with category metadata:

             product_id product_name         brand  discount_pct  return_rate
category                                                                     
Computing             1       Laptop          Dell            10          3.2
Accessories           2        Mouse      Logitech             5          1.5
Accessories           3     Keyboard            HP             5          1.5
Display               4      Monitor       Samsung             8          2.8
Accessories           5       Webcam     Zebronics             5          1.5
Accessories           6      USB Hub  AmazonBasics             5          1.5
Storage               7      SSD 1TB            WD             6          2.0
Audio                 8   Headphones          Sony            12          4.1


#### Observation:

- `df.join()` matched purely on index (`category` name).
- No `on=` needed — index alignment is automatic.
- Use `df.join()` when your lookup/reference table is **indexed by the key you want to join on**.
- It's cleaner than `pd.merge()` in those scenarios.

## Section 7: `indicator=True`: Where Did Each Row Come From?

Adding `indicator=True` to a merge creates a `_merge` column showing the source of each row.

| `_merge` value | Meaning |
|---|---|
| `left_only` | Row exists only in the LEFT table |
| `right_only` | Row exists only in the RIGHT table |
| `both` | Row matched in BOTH tables |

**This is extremely useful for:**
- Data quality checks (finding unmatched records)
- Debugging join logic
- Replicating SQL `EXCEPT` / `NOT IN` logic

In [67]:
# OUTER JOIN with indicator — full visibility of all rows
indicator_merge = pd.merge(
    orders, customers,
    on='customer_id',
    how='outer',
    indicator=True          # ← adds _merge column
)

print("Value counts of _merge column:")
print(indicator_merge['_merge'].value_counts())

print("\nFull breakdown:")
print(indicator_merge[['order_id', 'customer_id', 'name', '_merge']].to_string(index=False))

Value counts of _merge column:
_merge
both          14
right_only     2
left_only      1
Name: count, dtype: int64

Full breakdown:
 order_id  customer_id  name     _merge
    101.0            1  Amit       both
    103.0            1  Amit       both
    113.0            1  Amit       both
    102.0            2 Priya       both
    110.0            2 Priya       both
    104.0            3  Ravi       both
    111.0            3  Ravi       both
    105.0            4 Sneha       both
    106.0            5 Karan       both
    112.0            5 Karan       both
    107.0            6 Divya       both
    108.0            7 Rohit       both
    114.0            7 Rohit       both
    109.0            8 Meena       both
      NaN            9 Arjun right_only
      NaN           10 Pooja right_only
    115.0           99   NaN  left_only


In [69]:
# Practical use 1: Find orders with NO matching customer (data quality issue)
ghost_orders = indicator_merge[indicator_merge['_merge'] == 'left_only']
print("Orders with no customer record (ghost orders):")
print(ghost_orders[['order_id', 'customer_id', '_merge']])

# Practical use 2: Find customers who NEVER ordered (marketing targets)
inactive_customers = indicator_merge[indicator_merge['_merge'] == 'right_only']
print("\nCustomers with no orders (inactive — target for campaigns):")
print(inactive_customers[['customer_id', 'name', 'city', 'segment', '_merge']])

Orders with no customer record (ghost orders):
    order_id  customer_id     _merge
16     115.0           99  left_only

Customers with no orders (inactive — target for campaigns):
    customer_id   name       city    segment      _merge
14            9  Arjun  Ahmedabad     Retail  right_only
15           10  Pooja      Surat  Wholesale  right_only


#### Key Takeaway — `indicator=True`

- Think of `indicator=True` as a **join audit tool**.
- After any outer join, always check `_merge` value counts —
- unexpected `left_only` or `right_only` rows often signal
- dirty data, missing records, or a wrong join key.
- It answers *"how do you find records in Table A that don't exist in Table B"*
- It is SQL equivalent of `LEFT JOIN ... WHERE B.key IS NULL`

## Section 8: Many-to-Many Merges

A many-to-many merge happens when **both** tables have duplicate values in the join key.

Pandas performs a **cartesian product** on matching rows.

| Join Type | Left key | Right key | Result rows |
|---|---|---|---|
| One-to-one | Unique | Unique | Same as input |
| One-to-many | Unique | Duplicates | Expands |
| Many-to-many | Duplicates | Duplicates | Explodes (rows × rows) |

**Business Question:**

> Each order can have multiple items, and each product appears in multiple orders.
> 
> Merging `order_items` with `products` on `product_id` is a many-to-many scenario


In [75]:
# Many-to-many: order_items (20 rows) × products (8 rows) on product_id
# product_id repeats in order_items → one-to-many (product → items)

m2m = pd.merge(order_items, products, on='product_id', how='inner')

print(f"order_items rows : {len(order_items)}")
print(f"products rows    : {len(products)}")
print(f"After merge      : {len(m2m)}  ← each item enriched with product info")

print("\nSample output:")
print(m2m[['item_id', 'order_id', 'product_name', 'category',
           'quantity', 'unit_price']].head(8))

order_items rows : 20
products rows    : 8
After merge      : 20  ← each item enriched with product info

Sample output:
   item_id  order_id product_name     category  quantity  unit_price
0        1       101       Laptop    Computing         1       75000
1        2       101     Keyboard  Accessories         2        1500
2        3       102        Mouse  Accessories         1         500
3        4       103       Webcam  Accessories         3        3000
4        5       104       Laptop    Computing         1       75000
5        6       104      Monitor      Display         1       18000
6        7       105      USB Hub  Accessories         2        2500
7        8       106        Mouse  Accessories         4         500


In [6]:
# True many-to-many demo — tags system
# One product can have many tags, one tag can apply to many products
product_tags = pd.DataFrame({
    'product_id': [1, 1, 2, 2, 3, 3, 4],
    'tag':        ['premium', 'work-from-home', 
                   'office', 'work-from-home',
                   'office', 'ergonomic',
                   'work-from-home']
})

order_tags = pd.DataFrame({
    'order_id':   [101, 101, 104],
    'product_id': [1,   1,   1],
    'tag':        ['premium', 'work-from-home', 'premium']
})

print("product_tags rows:", len(product_tags))
print("order_tags rows  :", len(order_tags))

# Merge on BOTH product_id AND tag → true many-to-many
m2m_true = pd.merge(product_tags, order_tags, on=['product_id', 'tag'], how='inner')
print(f"\nAfter many-to-many merge: {len(m2m_true)} rows")
print(m2m_true)

print("\n⚠️  Row count is product of matching combinations — always verify this!")

product_tags rows: 7
order_tags rows  : 3

After many-to-many merge: 3 rows
   product_id             tag  order_id
0           1         premium       101
1           1         premium       104
2           1  work-from-home       101

⚠️  Row count is product of matching combinations — always verify this!


#### Key Takeaway -- Many to Many

> Always check row counts **before and after** a merge.
> An unexpected row explosion usually means:
> 1. Duplicate keys in one or both tables (unintended many-to-many)
> 2. Wrong join key chosen
> 3. Missing deduplication step upstream


**Quick sanity check pattern:**
```python
assert len(merged) <= max(len(df1), len(df2)) * some_factor, "Row explosion detected!"
```

## Section 9: Real World Multi-table Scenario

**Business Problem:**
The management team wants a single comprehensive report showing:

1. Total revenue per customer (name, city, segment)
2. Number of orders placed
3. Most purchased product category
4. Flag high-value customers (revenue > ₹1,00,000)

This requires joining all 4 tables:
`order_items` → `orders` → `customers` → `products`

In [18]:
# Step 1: order_items + products → enrich items with product info
items_products = pd.merge(
    order_items, products[['product_id', 'product_name', 'category']],
    on='product_id', how='left'
)

# Calculate line-level revenue
items_products['line_revenue'] = items_products['quantity'] * items_products['unit_price']

print("Step 1 — Items enriched with product info:")
print(items_products[['item_id', 'order_id', 'product_name', 
                       'category', 'quantity', 'unit_price', 'line_revenue']].head(5))

Step 1 — Items enriched with product info:
   item_id  order_id product_name     category  quantity  unit_price  \
0        1       101       Laptop    Computing         1       75000   
1        2       101     Keyboard  Accessories         2        1500   
2        3       102        Mouse  Accessories         1         500   
3        4       103       Webcam  Accessories         3        3000   
4        5       104       Laptop    Computing         1       75000   

   line_revenue  
0         75000  
1          3000  
2           500  
3          9000  
4         75000  


In [20]:
# Step 2: Add order info (customer_id, status, date)
items_orders = pd.merge(
    items_products, orders[['order_id', 'customer_id', 'order_date', 'status']],
    on='order_id', how='left'
)

print("Step 2 — Items with order context:")
print(items_orders[['item_id', 'order_id', 'customer_id', 
                     'product_name', 'line_revenue', 'status']].head(5))

Step 2 — Items with order context:
   item_id  order_id  customer_id product_name  line_revenue     status
0        1       101            1       Laptop         75000  Delivered
1        2       101            1     Keyboard          3000  Delivered
2        3       102            2        Mouse           500  Delivered
3        4       103            1       Webcam          9000    Shipped
4        5       104            3       Laptop         75000  Delivered


In [22]:
# Step 3: Add customer info
full_data = pd.merge(
    items_orders, customers,
    on='customer_id', how='left'
)

print("Step 3 — Full joined dataset shape:", full_data.shape)
print(full_data[['name', 'city', 'segment', 'product_name', 
                 'category', 'line_revenue', 'status']].head(5))

Step 3 — Full joined dataset shape: (20, 14)
    name       city    segment product_name     category  line_revenue  \
0   Amit      Delhi     Retail       Laptop    Computing         75000   
1   Amit      Delhi     Retail     Keyboard  Accessories          3000   
2  Priya     Mumbai  Wholesale        Mouse  Accessories           500   
3   Amit      Delhi     Retail       Webcam  Accessories          9000   
4   Ravi  Bangalore     Retail       Laptop    Computing         75000   

      status  
0  Delivered  
1  Delivered  
2  Delivered  
3    Shipped  
4  Delivered  


In [24]:
# Step 4: Build the final customer report

# Revenue and order count per customer
revenue_summary = full_data.groupby(
    ['customer_id', 'name', 'city', 'segment']
).agg(
    total_revenue   = ('line_revenue', 'sum'),
    total_orders    = ('order_id',     'nunique'),
    items_purchased = ('item_id',      'count')
).reset_index()

# Top category per customer
top_category = (
    full_data.groupby(['customer_id', 'category'])['line_revenue']
    .sum()
    .reset_index()
    .sort_values('line_revenue', ascending=False)
    .drop_duplicates(subset='customer_id')
    .rename(columns={'category': 'top_category', 'line_revenue': 'cat_revenue'})
)

# Final report
final_report = pd.merge(revenue_summary, 
                         top_category[['customer_id', 'top_category']], 
                         on='customer_id', how='left')

# Flag high-value customers
final_report['high_value'] = final_report['total_revenue'] > 100000

# Sort by revenue
final_report = final_report.sort_values('total_revenue', ascending=False)

print("=" * 65)
print("         CUSTOMER REVENUE REPORT — 2024")
print("=" * 65)
print(final_report.to_string(index=False))

         CUSTOMER REVENUE REPORT — 2024
 customer_id  name      city   segment  total_revenue  total_orders  items_purchased top_category  high_value
           2 Priya    Mumbai Wholesale         228000             2                3    Computing        True
           3  Ravi Bangalore    Retail         129000             2                3    Computing        True
           1  Amit     Delhi    Retail         103500             3                5    Computing        True
           7 Rohit   Kolkata Wholesale          13500             2                2        Audio       False
           6 Divya   Chennai    Retail           9500             1                2      Storage       False
           4 Sneha      Pune    Retail           5000             1                1  Accessories       False
           5 Karan Hyderabad Wholesale           3500             2                2  Accessories       False
           8 Meena    Jaipur    Retail           3000             1             

## Section 10: Summary and Key Interview Tips

**Functions Covered**

| Function | Use Case |
|---|---|
| `pd.merge(..., how='inner')` | Only matching rows |
| `pd.merge(..., how='left')` | Keep all left rows |
| `pd.merge(..., how='right')` | Keep all right rows |
| `pd.merge(..., how='outer')` | Keep everything |
| `pd.merge(..., on=[...])` | Multi-key join |
| `pd.merge(..., left_on=, right_on=)` | Different key names |
| `pd.merge(..., suffixes=)` | Handle column name clash |
| `pd.merge(..., indicator=True)` | Audit join results |
| `pd.concat([...], axis=0)` | Stack rows |
| `pd.concat([...], keys=[...])` | Stack with source label |
| `df.join()` | Index-based join |

#### Interview Tips

1. **Always check row counts** before and after a merge — unexpected
   changes signal wrong join type or duplicate keys.

2. **Default is inner join** — if rows are mysteriously disappearing,
   switch to `how='left'` and check for NaNs.

3. **`indicator=True`** is your best debugging tool for join issues.

4. **Prefer `suffixes=`** over default `_x/_y` in any production code.

5. **Multi-table joins** — always build them step by step,
   verifying shape at each stage rather than chaining everything at once.